In [0]:
import duckdb

In [0]:
with duckdb.connect('residentcalibration.duckdb') as con:
    # Set configuration for better performance

    # More efficient preview
    con.table("stg_transponderOwnership").show()

In [0]:
with duckdb.connect('files.duckdb') as con:
    con.sql("SELECT * FROM stg_trips ORDER BY trip_id ASC").show()


In [16]:
# Connect to DuckDB
con = duckdb.connect('files.duckdb')

# Install extensions
con.execute("INSTALL azure;")
con.execute("LOAD azure;")
con.execute("INSTALL delta;")
con.execute("LOAD delta;")

# Set up Azure authentication
con.execute("""
    CREATE OR REPLACE SECRET azure_secret (
        TYPE AZURE,
        CONNECTION_STRING 'AccountName=adlsdasadsdevwest;SharedAccessSignature=***REDACTED***'
    );
""")

# Read from Delta table
delta_table_path = "***REDACTED***"

# Query the Delta table
result = con.sql(f"""
    SELECT * 
    FROM delta_scan('{delta_table_path}')
""").show()

con.close()

#print(result)

# # Or load into a DuckDB table
# con.execute(f"""
#     CREATE TABLE local_table AS 
#     SELECT * FROM delta_scan('{delta_table_path}')
# """)

┌─────────────┬─────────────┬────────────┬────────┬────────────────────────┬───────────────────────────────┐
│ scenario_id │    model    │ trip_mode  │ count  │ loader__delta_hash_key │     loader__updated_date      │
│    int32    │   varchar   │  varchar   │ int64  │         int32          │   timestamp with time zone    │
├─────────────┼─────────────┼────────────┼────────┼────────────────────────┼───────────────────────────────┤
│        1150 │ visitor     │ WALK       │  62558 │            -1813879661 │ 2025-06-03 16:49:46.601516-07 │
│        1150 │ visitor     │ TNC_SINGLE │  58645 │              968227851 │ 2025-06-03 16:49:46.601516-07 │
│        1150 │ visitor     │ SHARED3    │  42819 │            -1374707706 │ 2025-06-03 16:49:46.601516-07 │
│        1150 │ visitor     │ TAXI       │   9132 │             1518431309 │ 2025-06-03 16:49:46.601516-07 │
│        1150 │ visitor     │ SHARED2    │   5661 │            -2025318090 │ 2025-06-03 16:49:46.601516-07 │
│        1150 │ vis